In [ ]:
import os
from copy import copy
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
 #Read reference BMB

refrun = '../hpcoutput/s18p1'
ds_ref = xr.open_dataset(os.path.join(refrun,'main_output_ANT_00001.nc'))
BMB_ref = ds_ref.sel(time=10000).drop_vars('time')
BMB_ref = BMB_ref.drop_vars(['Hi','Hb','Hs','SL','u_surf','v_surf','uabs_surf','mask','dHi','till_friction_angle','grounding_line'])
BMB_ref = BMB_ref.drop_vars(['calving_front','ice_margin','coastline','grounded_ice_contour'])
BMB_ref = BMB_ref.drop_vars(['zeta','depth','month'])
ds_ref.close()
del BMB_ref.encoding['unlimited_dims']

BMB_ref.to_netcdf(f'../input/larmip/BMB_ctrl.nc')

In [ ]:
im = plt.scatter(BMB_ref.V[0,:],BMB_ref.V[1,:],1,c=BMB_ref.BMB)#,vmin=-10,vmax=-6)
plt.colorbar(im)

In [ ]:
BMB_eais = BMB_ref.copy(deep=True)
BMB_ross = BMB_ref.copy(deep=True)
BMB_amun = BMB_ref.copy(deep=True)
BMB_wedd = BMB_ref.copy(deep=True)
BMB_apen = BMB_ref.copy(deep=True)

In [ ]:
 #Read basin masks

ds = xr.open_dataset('../input/larmip/LARMIP3_sectors.nc')
mask = ds.z
ds.close()

In [ ]:
#Set magnitude (m/yr)
mag = -8

In [ ]:
for i,vi in enumerate(BMB_ref.vi):
    xi = np.argmin((BMB_ref.V[0,i].values-mask.x.values)**2)
    yi = np.argmin((BMB_ref.V[1,i].values-mask.y.values)**2)
    maskval = int(mask[yi,xi].values)
    if maskval == 1:
        BMB_wedd.BMB[i] += mag
    elif maskval == 2:
        BMB_eais.BMB[i] += mag
    elif maskval == 3:
        BMB_ross.BMB[i] += mag
    elif maskval == 4:
        BMB_amun.BMB[i] += mag
    elif maskval == 5:
        BMB_apen.BMB[i] += mag

In [ ]:
BMB_eais.to_netcdf('../input/larmip/BMB_eais.nc')
BMB_ross.to_netcdf('../input/larmip/BMB_ross.nc')
BMB_amun.to_netcdf('../input/larmip/BMB_amun.nc')
BMB_wedd.to_netcdf('../input/larmip/BMB_wedd.nc')
BMB_apen.to_netcdf('../input/larmip/BMB_apen.nc')

In [ ]:
BMB_view = BMB_ref

im = plt.scatter(BMB_view.V[0,:],BMB_view.V[1,:],1,c=BMB_view.BMB)#,vmin=-10,vmax=-6)
plt.colorbar(im)